# MARKETING CAMPAIGN TREND ANALYSIS & ML PREDICTION
## Future Campaign Recommendations using Machine Learning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [2]:
# Load the same data as in the main analysis notebook
df = pd.read_excel("D:\\Downloads\\2024-05-marketing-campaign-analysis\\Marketing Campaign Analysis Dataset_May 2024.xlsx")
print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset Shape: (9900, 18)

Columns: ['Campaign', 'Date', 'City/Location', 'Latitude', 'Longitude', 'Channel', 'Device', 'Ad', 'Impressions', 'CTR, %', 'Clicks', 'Daily Average CPC', 'Spend, GBP', 'Conversions', 'Total conversion value, GBP', 'Likes (Reactions)', 'Shares', 'Comments']

First few rows:


,Campaign,Date,City/Location,Latitude,Longitude,Channel,Device,Ad,Impressions,"CTR, %",Clicks,Daily Average CPC,"Spend, GBP",Conversions,"Total conversion value, GBP",Likes (Reactions),Shares,Comments
0,Spring,2023-03-01,Birmingham,52.489471,-1.898575,Facebook,Desktop,Collection,1110.8,0.0185,20.54980,1.3915,28.595047,1,51.840,45.0,4.0,3.0
1,Spring,2023-03-02,Birmingham,52.489471,-1.898575,Facebook,Desktop,Collection,1296.6,0.0110,14.26260,0.7245,10.333254,2,40.320,14.0,17.0,10.5
2,Spring,2023-03-03,Birmingham,52.489471,-1.898575,Facebook,Desktop,Collection,1264.4,0.0132,16.69008,0.3795,6.333885,4,53.760,24.0,1.0,7.5
3,Spring,2023-03-04,Birmingham,52.489471,-1.898575,Facebook,Desktop,Collection,837.8,0.0159,13.32102,0.5750,7.659586,3,25.920,59.0,10.0,6.0
4,Spring,2023-03-05,Birmingham,52.489471,-1.898575,Facebook,Desktop,Collection,1599.0,0.0144,23.02560,0.8280,19.065197,2,108.192,44.0,17.0,1.5


# 1. TREND ANALYSIS BY CAMPAIGN, CHANNEL & DEVICE

In [3]:
print("="*80)
print("TREND ANALYSIS: WHAT WORKED IN OUR CAMPAIGNS")
print("="*80)

# Campaign Performance Trends
campaign_trends = df.groupby('Campaign').agg({
    'Impressions': ['sum', 'mean'],
    'Clicks': ['sum', 'mean'],
    'CTR, %': 'mean',
    'Conversions': ['sum', 'mean'],
    'Spend, GBP': 'sum',
    'Total conversion value, GBP': 'sum'
}).round(2)

campaign_trends.columns = ['_'.join(col).strip() for col in campaign_trends.columns.values]
print("\nCAMPAIGN PERFORMANCE SUMMARY:")
print(campaign_trends)

# Calculate ROI and Cost per Conversion
campaign_roi = df.groupby('Campaign').apply(lambda x: {
    'Total_Spend': x['Spend, GBP'].sum(),
    'Total_Conversions': x['Conversions'].sum(),
    'Total_Value': x['Total conversion value, GBP'].sum(),
    'Cost_Per_Conversion': x['Spend, GBP'].sum() / x['Conversions'].sum() if x['Conversions'].sum() > 0 else np.inf,
    'ROI': ((x['Total conversion value, GBP'].sum() - x['Spend, GBP'].sum()) / x['Spend, GBP'].sum() * 100) if x['Spend, GBP'].sum() > 0 else 0
}).apply(pd.Series).round(2)

print("\nCAMPAIGN ROI ANALYSIS:")
print(campaign_roi)

# Best performing campaign
best_campaign = campaign_roi['ROI'].idxmax()
print(f"\n✓ BEST PERFORMING CAMPAIGN: {best_campaign} with ROI of {campaign_roi.loc[best_campaign, 'ROI']:.2f}%")

TREND ANALYSIS: WHAT WORKED IN OUR CAMPAIGNS

CAMPAIGN PERFORMANCE SUMMARY:
          Impressions_sum  Impressions_mean  Clicks_sum  Clicks_mean  \
Campaign                                                               
Fall            6434259.0           1964.06    85106.33        25.98   
Spring          4751611.4           1434.67    57657.98        17.41   
Summer          3459578.4           1044.56    38821.71        11.72   

          CTR, %_mean  Conversions_sum  Conversions_mean  Spend, GBP_sum  \
Campaign                                                                   
Fall             0.01            14886              4.54        79313.93   
Spring           0.01            12613              3.81        49554.52   
Summer           0.01            12753              3.85        34381.61   

          Total conversion value, GBP_sum  
Campaign                                   
Fall                            749006.03  
Spring                          498285.14  
Summer

In [4]:
# Channel Performance Trends
print("\n" + "="*80)
print("CHANNEL PERFORMANCE ANALYSIS")
print("="*80)

channel_performance = df.groupby('Channel').apply(lambda x: {
    'Total_Impressions': x['Impressions'].sum(),
    'Total_Clicks': x['Clicks'].sum(),
    'Avg_CTR': x['CTR, %'].mean(),
    'Total_Conversions': x['Conversions'].sum(),
    'Total_Spend': x['Spend, GBP'].sum(),
    'Cost_Per_Click': x['Spend, GBP'].sum() / x['Clicks'].sum() if x['Clicks'].sum() > 0 else np.inf,
    'Cost_Per_Conversion': x['Spend, GBP'].sum() / x['Conversions'].sum() if x['Conversions'].sum() > 0 else np.inf,
    'Conversion_Rate': (x['Conversions'].sum() / x['Clicks'].sum() * 100) if x['Clicks'].sum() > 0 else 0
}).apply(pd.Series).round(2)

print("\nCHANNEL PERFORMANCE METRICS:")
print(channel_performance)

# Best and worst channels
best_channel_ctr = channel_performance['Avg_CTR'].idxmax()
best_channel_conv = channel_performance['Conversion_Rate'].idxmax()
best_channel_cpc = channel_performance['Cost_Per_Click'].idxmin()

print(f"\n✓ BEST CTR CHANNEL: {best_channel_ctr} ({channel_performance.loc[best_channel_ctr, 'Avg_CTR']:.2f}%)")
print(f"✓ BEST CONVERSION RATE CHANNEL: {best_channel_conv} ({channel_performance.loc[best_channel_conv, 'Conversion_Rate']:.2f}%)")
print(f"✓ LOWEST CPC CHANNEL: {best_channel_cpc} (£{channel_performance.loc[best_channel_cpc, 'Cost_Per_Click']:.2f})")


CHANNEL PERFORMANCE ANALYSIS

CHANNEL PERFORMANCE METRICS:
           Total_Impressions  Total_Clicks  Avg_CTR  Total_Conversions  \
Channel                                                                  
Facebook           5439590.6      69968.71     0.01            13132.0   
Instagram          4840638.1      68605.64     0.01            15590.0   
Pinterest          4365220.1      43011.67     0.01            11530.0   

           Total_Spend  Cost_Per_Click  Cost_Per_Conversion  Conversion_Rate  
Channel                                                                       
Facebook      71612.54            1.02                 5.45            18.77  
Instagram     63394.01            0.92                 4.07            22.72  
Pinterest     28243.52            0.66                 2.45            26.81  

✓ BEST CTR CHANNEL: Facebook (0.01%)
✓ BEST CONVERSION RATE CHANNEL: Pinterest (26.81%)
✓ LOWEST CPC CHANNEL: Pinterest (£0.66)


In [5]:
# Device Performance Trends
print("\n" + "="*80)
print("DEVICE PERFORMANCE ANALYSIS")
print("="*80)

device_performance = df.groupby('Device').apply(lambda x: {
    'Total_Impressions': x['Impressions'].sum(),
    'Total_Clicks': x['Clicks'].sum(),
    'Avg_CTR': x['CTR, %'].mean(),
    'Total_Conversions': x['Conversions'].sum(),
    'Total_Spend': x['Spend, GBP'].sum(),
    'Cost_Per_Conversion': x['Spend, GBP'].sum() / x['Conversions'].sum() if x['Conversions'].sum() > 0 else np.inf,
    'Avg_Conversion_Value': x['Total conversion value, GBP'].sum() / x['Conversions'].sum() if x['Conversions'].sum() > 0 else 0
}).apply(pd.Series).round(2)

print("\nDEVICE PERFORMANCE METRICS:")
print(device_performance)

best_device = device_performance['Avg_CTR'].idxmax()
print(f"\n✓ BEST PERFORMING DEVICE: {best_device} with Avg CTR of {device_performance.loc[best_device, 'Avg_CTR']:.2f}%")


DEVICE PERFORMANCE ANALYSIS

DEVICE PERFORMANCE METRICS:
         Total_Impressions  Total_Clicks  Avg_CTR  Total_Conversions  \
Device                                                                 
Desktop          5800159.8      88849.20     0.01            21310.0   
Mobile           8845289.0      92736.82     0.01            18942.0   

         Total_Spend  Cost_Per_Conversion  Avg_Conversion_Value  
Device                                                           
Desktop     86218.78                 4.05                 44.57  
Mobile      77031.28                 4.07                 41.28  

✓ BEST PERFORMING DEVICE: Desktop with Avg CTR of 0.01%


In [6]:
# Interactive Trend Visualization - Campaign Trends
campaign_summary = df.groupby('Campaign').agg({
    'CTR, %': 'mean',
    'Conversions': 'mean',
    'Spend, GBP': 'mean'
}).reset_index()

fig_campaign_trends = go.Figure()

fig_campaign_trends.add_trace(go.Bar(
    x=campaign_summary['Campaign'],
    y=campaign_summary['CTR, %'],
    name='Avg CTR (%)',
    marker_color='indianred'
))

fig_campaign_trends.update_layout(
    title='Campaign Performance Trends - Click-Through Rate',
    xaxis_title='Campaign',
    yaxis_title='Average CTR (%)',
    height=500,
    template='plotly_white',
    font=dict(size=12)
)
fig_campaign_trends.show()

# Channel Performance Visualization
channel_summary = df.groupby('Channel').agg({
    'CTR, %': 'mean',
    'Conversions': 'mean',
    'Spend, GBP': 'mean'
}).reset_index().sort_values('CTR, %', ascending=False)

fig_channel_trends = px.bar(channel_summary, x='Channel', y='CTR, %',
                             title='Channel Performance - Average CTR by Channel',
                             color='CTR, %',
                             color_continuous_scale='Viridis',
                             labels={'CTR, %': 'Average CTR (%)'},
                             template='plotly_white')
fig_channel_trends.update_layout(height=500, showlegend=False, font=dict(size=12))
fig_channel_trends.show()

print("✓ Trend visualizations displayed")

✓ Trend visualizations displayed


# 2. PREDICTIVE MODELING - CONVERSION PREDICTION

In [7]:
# Data Preparation for ML Models
df_ml = df.copy()

# Encode categorical variables
label_encoders = {}
categorical_features = ['Campaign', 'City/Location', 'Channel', 'Device', 'Ad']

for col in categorical_features:
    if col in df_ml.columns:
        le = LabelEncoder()
        df_ml[col + '_encoded'] = le.fit_transform(df_ml[col])
        label_encoders[col] = le

# Select features for prediction
feature_cols = ['Latitude', 'Longitude', 'Impressions', 'CTR, %', 'Clicks', 
                 'Daily Average CPC', 'Spend, GBP', 'Campaign_encoded', 
                 'Channel_encoded', 'Device_encoded']

# Remove rows with missing target variable
df_ml = df_ml.dropna(subset=['Conversions'])

X = df_ml[feature_cols]
y = df_ml['Conversions']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("="*80)
print("MACHINE LEARNING MODELS - CONVERSION PREDICTION")
print("="*80)
print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"Total features: {len(feature_cols)}")

MACHINE LEARNING MODELS - CONVERSION PREDICTION

Training set size: 7920
Testing set size: 1980
Total features: 10


In [8]:
# Train Multiple ML Models
models = {}
results = []

# Model 1: Linear Regression
print("\nTraining Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_r2 = r2_score(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae = mean_absolute_error(y_test, lr_pred)
models['Linear Regression'] = lr_model
results.append({'Model': 'Linear Regression', 'R² Score': lr_r2, 'RMSE': lr_rmse, 'MAE': lr_mae})
print(f"  ✓ R² Score: {lr_r2:.4f}, RMSE: {lr_rmse:.4f}")

# Model 2: Random Forest
print("\nTraining Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)
models['Random Forest'] = rf_model
results.append({'Model': 'Random Forest', 'R² Score': rf_r2, 'RMSE': rf_rmse, 'MAE': rf_mae})
print(f"  ✓ R² Score: {rf_r2:.4f}, RMSE: {rf_rmse:.4f}")

# Model 3: Gradient Boosting
print("\nTraining Gradient Boosting...")
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_r2 = r2_score(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_mae = mean_absolute_error(y_test, gb_pred)
models['Gradient Boosting'] = gb_model
results.append({'Model': 'Gradient Boosting', 'R² Score': gb_r2, 'RMSE': gb_rmse, 'MAE': gb_mae})
print(f"  ✓ R² Score: {gb_r2:.4f}, RMSE: {gb_rmse:.4f}")

# Results comparison
results_df = pd.DataFrame(results).round(4)
print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON")
print("="*80)
print(results_df.to_string(index=False))

best_model_name = results_df.loc[results_df['R² Score'].idxmax(), 'Model']
print(f"\n✓ BEST MODEL: {best_model_name}")


Training Linear Regression...
  ✓ R² Score: 0.0384, RMSE: 2.2377

Training Random Forest...
  ✓ R² Score: 0.0943, RMSE: 2.1717

Training Gradient Boosting...
  ✓ R² Score: 0.1447, RMSE: 2.1104

MODEL PERFORMANCE COMPARISON
            Model  R² Score   RMSE    MAE
Linear Regression    0.0384 2.2377 1.8584
    Random Forest    0.0943 2.1717 1.7814
Gradient Boosting    0.1447 2.1104 1.7601

✓ BEST MODEL: Gradient Boosting


In [9]:
# Feature Importance Analysis (Random Forest)
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance (Top 10):")
print(feature_importance.head(10).to_string(index=False))

# Interactive Feature Importance Plot
fig_importance = px.bar(feature_importance.head(10), x='Importance', y='Feature',
                         title='Top 10 Most Important Features for Conversion Prediction',
                         orientation='h',
                         color='Importance',
                         color_continuous_scale='Blues',
                         template='plotly_white')
fig_importance.update_layout(height=500, showlegend=False, font=dict(size=11))
fig_importance.update_xaxes(title_text='Importance Score')
fig_importance.update_yaxes(title_text='Feature')
fig_importance.show()

print("\n✓ Feature importance visualization displayed")


FEATURE IMPORTANCE ANALYSIS

Feature Importance (Top 10):
          Feature  Importance
           Clicks    0.165068
      Impressions    0.162447
Daily Average CPC    0.159315
       Spend, GBP    0.150053
           CTR, %    0.148511
 Campaign_encoded    0.068642
  Channel_encoded    0.052081
   Device_encoded    0.041211
        Longitude    0.027139
         Latitude    0.025534



✓ Feature importance visualization displayed


# 3. ACTUAL vs PREDICTED PERFORMANCE

In [10]:
# Platform-Specific Content & Budget Recommendations
print("\n" + "="*80)
print("PLATFORM-SPECIFIC RECOMMENDATIONS")
print("="*80)

if 'Channel' in df.columns:
    platform_budget = df.groupby('Channel').apply(lambda x: {
        'Current_Budget': x['Spend, GBP'].sum(),
        'Current_Conversions': x['Conversions'].sum(),
        'Avg_CTR': x['CTR, %'].mean(),
        'Conversion_Rate': (x['Conversions'].sum() / x['Clicks'].sum() * 100) if x['Clicks'].sum() > 0 else 0,
        'ROI': ((x['Total conversion value, GBP'].sum() - x['Spend, GBP'].sum()) / x['Spend, GBP'].sum() * 100) if x['Spend, GBP'].sum() > 0 else 0
    }).apply(pd.Series).round(2)
    
    print("\nChannel Analysis:")
    print(platform_budget)
    
    total_budget = df['Spend, GBP'].sum()
    
    # Recommend budget allocation based on ROI
    print("\nRECOMMENDED BUDGET ALLOCATION (Based on ROI & Conversion Rate):")
    
    platform_budget['Recommended_Allocation_%'] = (platform_budget['ROI'] / platform_budget['ROI'].sum() * 100).round(1)
    platform_budget['Recommended_Budget'] = (platform_budget['Recommended_Allocation_%'] / 100 * total_budget).round(0)
    
    budget_allocation = platform_budget[['Conversion_Rate', 'Recommended_Allocation_%', 'Recommended_Budget']].copy()
    budget_allocation.columns = ['Current_Conv_Rate_%', 'Recommended_%', 'Recommended_Budget_GBP']
    print(budget_allocation)
    
    # Interactive Budget Allocation Chart
    alloc_data = pd.DataFrame({
        'Channel': platform_budget.index,
        'Recommended_Budget': platform_budget['Recommended_Budget']
    })
    
    fig_budget = px.pie(alloc_data, values='Recommended_Budget', names='Channel',
                        title='Recommended Budget Allocation by Channel',
                        template='plotly_white')
    fig_budget.update_layout(height=500, font=dict(size=12))
    fig_budget.show()
    
    print("\n✓ Budget allocation visualization displayed")


PLATFORM-SPECIFIC RECOMMENDATIONS

Channel Analysis:
           Current_Budget  Current_Conversions  Avg_CTR  Conversion_Rate  \
Channel                                                                    
Facebook         71612.54              13132.0     0.01            18.77   
Instagram        63394.01              15590.0     0.01            22.72   
Pinterest        28243.52              11530.0     0.01            26.81   

               ROI  
Channel             
Facebook    475.63  
Instagram   980.17  
Pinterest  2147.29  

RECOMMENDED BUDGET ALLOCATION (Based on ROI & Conversion Rate):
           Current_Conv_Rate_%  Recommended_%  Recommended_Budget_GBP
Channel                                                              
Facebook                 18.77           13.2                 21549.0
Instagram                22.72           27.2                 44404.0
Pinterest                26.81           59.6                 97297.0



✓ Budget allocation visualization displayed


# 4. CAMPAIGN RECOMMENDATIONS & FUTURE STRATEGY

In [11]:
# Content Recommendations by Ad Type
print("\n" + "="*80)
print("CONTENT & AD TYPE RECOMMENDATIONS")
print("="*80)

if 'Ad' in df.columns:
    ad_performance_details = df.groupby('Ad').apply(lambda x: {
        'Total_Impressions': x['Impressions'].sum(),
        'Total_Clicks': x['Clicks'].sum(),
        'Avg_CTR': x['CTR, %'].mean(),
        'Total_Conversions': x['Conversions'].sum(),
        'Conversion_Rate': (x['Conversions'].sum() / x['Clicks'].sum() * 100) if x['Clicks'].sum() > 0 else 0,
        'Total_Spend': x['Spend, GBP'].sum(),
        'Total_Value': x['Total conversion value, GBP'].sum()
    }).apply(pd.Series).round(2)
    
    print("\nAd Type Performance Metrics:")
    print(ad_performance_details)
    
    best_ad = ad_performance_details['Avg_CTR'].idxmax()
    print(f"\n✓ BEST PERFORMING AD TYPE: {best_ad}")
    print(f"   - Average CTR: {ad_performance_details.loc[best_ad, 'Avg_CTR']:.2f}%")
    print(f"   - Conversion Rate: {ad_performance_details.loc[best_ad, 'Conversion_Rate']:.2f}%")
    
    print("\nCONTENT RECOMMENDATIONS:")
    print(f"   1. Prioritize '{best_ad}' ad content")
    print(f"   2. Scale successful creative formats")
    print(f"   3. A/B test variations of top-performing ads")
    print(f"   4. Allocate 60% of budget to {best_ad} ads")
    print(f"   5. Monitor performance metrics weekly")


CONTENT & AD TYPE RECOMMENDATIONS

Ad Type Performance Metrics:
            Total_Impressions  Total_Clicks  Avg_CTR  Total_Conversions  \
Ad                                                                        
Collection          7926327.1     105968.52     0.01            19069.0   
Discount            6719121.7      75617.50     0.01            21183.0   

            Conversion_Rate  Total_Spend  Total_Value  
Ad                                                     
Collection            17.99     72175.77    657638.92  
Discount              28.01     91074.30   1074061.52  

✓ BEST PERFORMING AD TYPE: Collection
   - Average CTR: 0.01%
   - Conversion Rate: 17.99%

CONTENT RECOMMENDATIONS:
   1. Prioritize 'Collection' ad content
   2. Scale successful creative formats
   3. A/B test variations of top-performing ads
   4. Allocate 60% of budget to Collection ads
   5. Monitor performance metrics weekly


In [12]:
# Platform-Specific Content & Budget Recommendations
print("\n" + "="*80)
print("PLATFORM-SPECIFIC RECOMMENDATIONS")
print("="*80)

if 'Channel' in df.columns:
    platform_budget = df.groupby('Channel').apply(lambda x: {
        'Current_Budget': x['Spend'].sum(),
        'Current_Conversions': x['Conversions'].sum(),
        'Avg_CTR': x['CTR'].mean(),
        'Conversion_Rate': (x['Conversions'].sum() / x['Clicks'].sum() * 100) if x['Clicks'].sum() > 0 else 0,
        'ROI': ((x['Total conversion value'].sum() - x['Spend'].sum()) / x['Spend'].sum() * 100) if x['Spend'].sum() > 0 else 0
    }).apply(pd.Series).round(2)
    
    print("\nChannel Analysis:")
    print(platform_budget)
    
    total_budget = df['Spend'].sum()
    
    # Recommend budget allocation based on ROI
    print("\nRECOMMENDED BUDGET ALLOCATION (Based on ROI & Conversion Rate):")
    
    platform_budget['Recommended_Allocation_%'] = (platform_budget['ROI'] / platform_budget['ROI'].sum() * 100).round(1)
    platform_budget['Recommended_Budget'] = (platform_budget['Recommended_Allocation_%'] / 100 * total_budget).round(0)
    
    budget_allocation = platform_budget[['Conversion_Rate', 'Recommended_Allocation_%', 'Recommended_Budget']].copy()
    budget_allocation.columns = ['Current_Conv_Rate_%', 'Recommended_%', 'Recommended_Budget_$']
    print(budget_allocation)
    
    # Interactive Budget Allocation Chart
    alloc_data = pd.DataFrame({
        'Channel': platform_budget.index,
        'Recommended_Budget': platform_budget['Recommended_Budget']
    })
    
    fig_budget = px.pie(alloc_data, values='Recommended_Budget', names='Channel',
                        title='Recommended Budget Allocation by Channel',
                        template='plotly_white')
    fig_budget.update_layout(height=500, font=dict(size=12))
    fig_budget.show()
    
    print("\n✓ Budget allocation visualization displayed")


PLATFORM-SPECIFIC RECOMMENDATIONS


KeyError: 'Spend'

In [ ]:
# Content Recommendations by Ad Type
print("\n" + "="*80)
print("CONTENT & AD TYPE RECOMMENDATIONS")
print("="*80)

if 'Ad' in df.columns:
    ad_performance_details = df.groupby('Ad').apply(lambda x: {
        'Total_Impressions': x['Impressions'].sum(),
        'Total_Clicks': x['Clicks'].sum(),
        'Avg_CTR': x['CTR'].mean(),
        'Total_Conversions': x['Conversions'].sum(),
        'Conversion_Rate': (x['Conversions'].sum() / x['Clicks'].sum() * 100) if x['Clicks'].sum() > 0 else 0,
        'Total_Spend': x['Spend'].sum(),
        'Total_Value': x['Total conversion value'].sum()
    }).apply(pd.Series).round(2)
    
    print("\nAd Type Performance Metrics:")
    print(ad_performance_details)
    
    best_ad = ad_performance_details['Avg_CTR'].idxmax()
    print(f"\n✓ BEST PERFORMING AD TYPE: {best_ad}")
    print(f"   - Average CTR: {ad_performance_details.loc[best_ad, 'Avg_CTR']:.2f}%")
    print(f"   - Conversion Rate: {ad_performance_details.loc[best_ad, 'Conversion_Rate']:.2f}%")
    
    print("\nCONTENT RECOMMENDATIONS:")
    print(f"   1. Prioritize '{best_ad}' ad content")
    print(f"   2. Scale successful creative formats")
    print(f"   3. A/B test variations of top-performing ads")
    print(f"   4. Allocate 60% of budget to {best_ad} ads")
    print(f"   5. Monitor performance metrics weekly")


CONTENT & AD TYPE RECOMMENDATIONS


KeyError: 'CTR'

In [ ]:
# Summary Dashboard
print("\n" + "="*80)
print("EXECUTIVE SUMMARY DASHBOARD")
print("="*80)

summary_data = {
    'Metric': [
        'Total Dataset Size',
        'Total Impressions',
        'Total Clicks',
        'Total Conversions',
        'Total Spend (GBP)',
        'Overall CTR',
        'Overall Conversion Rate',
        'Average Cost Per Conversion',
        'Total Conversion Value',
        'Overall ROI'
    ],
    'Value': [
        f"{len(df):,} records",
        f"{df['Impressions'].sum():,.0f}",
        f"{df['Clicks'].sum():,.0f}",
        f"{df['Conversions'].sum():,.0f}",
        f"£{df['Spend, GBP'].sum():,.2f}",
        f"{df['CTR, %'].mean():.2f}%",
        f"{(df['Conversions'].sum() / df['Clicks'].sum() * 100):.2f}%",
        f"£{df['Spend, GBP'].sum() / df['Conversions'].sum():.2f}" if df['Conversions'].sum() > 0 else "N/A",
        f"£{df['Total conversion value, GBP'].sum():,.2f}",
        f"{((df['Total conversion value, GBP'].sum() - df['Spend, GBP'].sum()) / df['Spend, GBP'].sum() * 100):.2f}%"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print("\n✓ All trend analysis and ML predictions completed successfully!")
print("✓ Ready for presentation and strategic decision-making")


EXECUTIVE SUMMARY DASHBOARD


                     Metric         Value
         Total Dataset Size 9,900 records
          Total Impressions    14,645,449
               Total Clicks       181,586
          Total Conversions        40,252
          Total Spend (GBP)   £163,250.07
                Overall CTR         0.01%
    Overall Conversion Rate        22.17%
Average Cost Per Conversion         £4.06
     Total Conversion Value £1,731,700.44
                Overall ROI       960.77%

ANALYSIS COMPLETE

✓ All trend analysis and ML predictions completed successfully!
✓ Ready for presentation and strategic decision-making
